# QUY TRÌNH ETL & XÂY DỰNG DATA WAREHOUSE (STAR SCHEMA)
## Phân tích hiệu suất quảng cáo toàn cầu (Global Ads Performance) - NEXORA

Notebook này thực hiện toàn diện quy trình **ETL (Extract - Transform - Load)** từ file CSV thô vào cơ sở dữ liệu **PostgreSQL**:
1. **Extract**: Đọc dữ liệu CSV nguồn và kiểm tra cấu trúc.
2. **Transform**: Ép kiểu số chuẩn (`Decimal`), tính nhãn `Spend Level` (`Low`/`Medium`/`High`), ghi nhận kiểm toán dữ liệu.
3. **Schema DDL**: Tạo mô hình hình sao (Star Schema) gồm **5 Bảng Dimension** và **1 Bảng Fact**.
4. **Load**: Nạp dữ liệu vào PostgreSQL.
5. **Validation**: Kiểm toán tự động số dòng và tổng tiền trước khi công bố.
6. **Export / Serving Views**: Xuất các View phục vụ trực tiếp cho **Dashboard BI** và **Hệ thống hỗ trợ ra quyết định (DSS)**.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

# Them thu muc goc de goi cac tien ich
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Import pipeline ETL tong hop
from etl_pipeline import (
    CONFIG_THRESHOLDS,
    CSV_PATH,
    extract_data,
    get_db_engine,
    transform_data,
)

engine = get_db_engine()
print("Ket noi PostgreSQL thanh cong!")

### Bước 1: Trích xuất dữ liệu gốc (Extract)
Đọc dữ liệu thô từ file CSV, tính mã băm SHA-256 để định danh phiên bản nguồn.

In [ ]:
extracted = extract_data(CSV_PATH)
df_raw = pd.DataFrame(extracted["rows"])

print(f"Ten file: {extracted['name']}")
print(f"Ma hash SHA-256: {extracted['checksum']}")
print(f"Tong so quan sat: {len(df_raw):,} dong, {len(df_raw.columns)} cot")
df_raw.head()

### Bước 2: Biến đổi & Kiểm tra chất lượng (Transform)
- Ép kiểu ngày `Date`, kiểu số nguyên `BIGINT` cho số lượt hiển thị, nhấp, chuyển đổi.
- Ép kiểu tiền tệ chính xác `NUMERIC(18,2)` (`Decimal`) cho `ad_spend` và `revenue`.
- Tự động dò ngưỡng theo từng cặp `Country` × `Industry` để gắn nhãn `Spend Level` (`Low`, `Medium`, `High`).
- Ghi nhận các cảnh báo sai lệch tỷ lệ CTR gốc.

In [ ]:
transformed = transform_data(extracted, CONFIG_THRESHOLDS)
df_clean = pd.DataFrame(transformed["rows"])
df_enriched = pd.DataFrame(transformed["enriched"])

print(f"So dong hop le: {len(df_clean):,} dong")
print(f"So canh bao chat luong: {len(transformed['issues'])} canh bao")

# Xem ty le phan bo Spend Level
print("\nPhan phoi Spend Level tren tap du lieu:")
print(df_enriched['spend_level'].value_counts())

df_clean.head()

### Bước 3 & 4: Xây dựng Star Schema & Nạp vào Database (Load)
Chạy quy trình nạp hoàn chỉnh: tạo bảng Dimension, bảng Fact, và nạp dữ liệu vào PostgreSQL.

In [ ]:
from etl_pipeline import run_etl

# Chay toan bo quy trinh nap du lieu
run_etl()

### Bước 5: Truy vấn và xuất các Bảng & View phân tích
Dưới đây là các bảng và view đã xuất bản sẵn sàng trong PostgreSQL để kết nối sang PowerBI / Metabase / DSS.

In [ ]:
# 1. Bang Fact phuc vu BI
with engine.connect() as conn:
    df_fact = pd.read_sql("SELECT * FROM mart.v_fact_ad_performance_current LIMIT 5;", conn)
print("=== 1. BANG FACT HIEN HANH (mart.v_fact_ad_performance_current) ===")
display(df_fact)

# 2. Bang Fact ghep san 5 Dimension & Spend Level
with engine.connect() as conn:
    df_analysis = pd.read_sql("SELECT date, platform, campaign_type, industry, country, impressions, clicks, ad_spend, conversions, revenue, spend_level FROM mart.v_ad_performance_analysis_current LIMIT 5;", conn)
print("\n=== 2. BANG PHAN TICH DA CHIEU (mart.v_ad_performance_analysis_current) ===")
display(df_analysis)

# 3. Bang DSS: Tong hop theo Thang x Nen tang (Month x Platform)
with engine.connect() as conn:
    df_dss = pd.read_sql("SELECT month_start, platform, observations, ad_spend, revenue, revenue_per_unit_spend, conversions_per_unit_spend FROM mart.v_platform_month_current ORDER BY month_start, platform LIMIT 6;", conn)
print("\n=== 3. BANG TONG HOP PHUC VU DSS (mart.v_platform_month_current) ===")
display(df_dss)

### Bước 6: Kiểm toán & Thống kê tổng hợp
Xác nhận các đại lượng tổng khớp tuyệt đối với chỉ số của Lớp ngữ nghĩa (Semantic Layer).

In [ ]:
with engine.connect() as conn:
    summary_query = text("""
        SELECT
            COUNT(*)::bigint AS "Tong so quan sat",
            SUM(impressions)::bigint AS "Tong Impressions",
            SUM(clicks)::bigint AS "Tong Clicks",
            SUM(ad_spend)::numeric(18,2) AS "Tong Chi phi",
            SUM(conversions)::bigint AS "Tong Chuyen doi",
            SUM(revenue)::numeric(18,2) AS "Tong Doanh thu",
            ROUND(SUM(clicks)::numeric / SUM(impressions) * 100, 2) AS "CTR Tong (%)",
            ROUND(SUM(ad_spend)::numeric / SUM(clicks), 2) AS "CPC Tong",
            ROUND(SUM(revenue)::numeric / SUM(ad_spend), 3) AS "ROAS Tong (lan)"
        FROM mart.v_fact_ad_performance_current;
    """)
    df_summary = pd.read_sql(summary_query, conn)

print("=== BANG DOI CHIEU TOAN BO CHI SO DATA WAREHOUSE ===")
display(df_summary)